# Proyecto 3: Mejorando algoritmos con CUDA
Runner de Colab para las fases con GPU.

**Antes de empezar:** Runtime > Change runtime type > **T4 GPU**.

## 0. Evidencia de hardware

In [ ]:
!nvidia-smi

## 1. Obtener el codigo
Clona el repositorio (reemplaza la URL). Alternativa: sube la carpeta `Proyecto3-paralela` al panel de archivos de Colab.

In [ ]:
# !git clone <URL_DEL_REPO>
%cd Proyecto3-paralela

## 2. Obtener el dataset (Kaggle grassknoted/asl-alphabet)

In [ ]:
import os, pathlib, kagglehub
ruta = kagglehub.dataset_download("grassknoted/asl-alphabet")
# el dataset anida asl_alphabet_train/asl_alphabet_train: buscamos la carpeta que contiene directamente las clases
train = next(p for p in pathlib.Path(ruta).rglob("*")
             if p.is_dir() and (p / "A").is_dir() and (p / "nothing").is_dir())
os.environ["ASL_DATA"] = str(train)
print(os.environ["ASL_DATA"])

## 3. FASE 1: preparacion de imagenes, CPU multinucleo
Compara el bucle secuencial contra `multiprocessing.Pool`. **Toma una captura del uso de CPU** (panel Resources de Colab o la celda de `top` de abajo, ejecutada mientras corre la fase).

In [ ]:
!python src/01_preparacion.py

In [ ]:
!top -bn1 | head -15

## 4. FASE 2: entrenamiento CPU vs GPU (CUDA/cuDNN)
Primero se lanza `nvidia-smi` en segundo plano para registrar la utilizacion de la GPU. **Toma una captura de `nvidia-smi` durante el entrenamiento.**

In [ ]:
!mkdir -p resultados
!nvidia-smi --query-gpu=timestamp,utilization.gpu,memory.used --format=csv -l 2 > resultados/nvidia_smi_log.csv &

In [ ]:
!python src/02_entrenamiento.py

In [ ]:
!pkill nvidia-smi; tail -20 resultados/nvidia_smi_log.csv

## 5. FASE 3: kernel CUDA propio con Numba

In [ ]:
!python src/03_kernel_cuda.py

## 6. Comparacion: tabla y graficas

In [ ]:
!python src/04_comparacion.py

In [ ]:
from IPython.display import Markdown, Image, display
import pathlib
display(Markdown(pathlib.Path("resultados/tabla_comparativa.md").read_text()))
for png in sorted(pathlib.Path("resultados/figuras").glob("*.png")):
    display(Image(str(png)))

## 7. Descargar resultados

In [ ]:
!zip -r resultados.zip resultados
from google.colab import files
files.download("resultados.zip")